# Gaussian Mixture Models (GMM)
**Autor:** Autoagente de Notebooks
**Fecha:** 2026-06-14
**Tags:** Gaussian Mixture Models, Clustering, EM, ML
**Propósito:** Explicar los fundamentos y la implementación del algoritmo EM para modelos de mezcla gaussiana.


## Descripción
Los Gaussian Mixture Models (GMM) son un método probabilístico para agrupar datos cuando se asume que cada subpoblación puede describirse mediante una distribución normal. A diferencia de k-means, que usa solo las medias de los grupos, GMM también modela la varianza de cada componente y produce asignaciones suaves de pertenencia.

Este notebook presenta la intuición de GMM, su formulación matemática con el algoritmo de Expectation-Maximization (EM) y un ejemplo práctico usando `scikit-learn` en datos sintéticos.


## Objetivos
- Comprender el problema que resuelve un Gaussian Mixture Model y cuándo es apropiado usarlo.
- Explicar la estructura del modelo y las ecuaciones del algoritmo EM.
- Implementar un ejemplo práctico de ajuste y visualización con datos sintéticos.


## Estructura del modelo
- Cada punto de datos se asume generado por una mezcla de $K$ distribuciones gaussianas.
- Cada componente tiene parámetros: proporción $\pi_k$, media $\mu_k$ y covarianza $\Sigma_k$.
- El modelo estima responsabilidades $r_{ik}$, que son probabilidades suaves de pertenencia.
- El algoritmo EM alterna entre estimar responsabilidades (E-step) y actualizar parámetros (M-step).


## Modelo matemático
Sea $x \in \mathbb{R}^D$ un vector de características. La densidad de un GMM se define como:
$$
p(x) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(x \mid \mu_k, \Sigma_k),
$$
con restricciones $\pi_k \ge 0$ y $\sum_{k=1}^K \pi_k = 1$.

La densidad de cada componente es:
$$
\mathcal{N}(x \mid \mu_k, \Sigma_k) = \frac{1}{(2\pi)^{D/2} |\Sigma_k|^{1/2}} \exp\left(-\frac{1}{2}(x-\mu_k)^T \Sigma_k^{-1} (x-\mu_k)\right).
$$

El algoritmo EM usa responsabilidades:
$$
r_{ik} = \frac{\pi_k\,\mathcal{N}(x_i\mid \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j\,\mathcal{N}(x_i\mid \mu_j, \Sigma_j)}.
$$

Modelar la suma suave de probabilidades:
$$
N_k^{\text{soft}} = \sum_{i=1}^{N} r_{ik}.
$$

Actualizaciones en el paso M:
$$
\mu_k = \frac{1}{N_k^{\text{soft}}} \sum_{i=1}^{N} r_{ik} \, x_i,
$$
$$
\Sigma_k = \frac{1}{N_k^{\text{soft}}} \sum_{i=1}^{N} r_{ik} \, (x_i - \mu_k)(x_i - \mu_k)^T,
$$
$$
\pi_k = \frac{N_k^{\text{soft}}}{N}.
$$

La convergencia se evalúa con la verosimilitud de los datos:
$$
\mathcal{L}(\theta) = \sum_{i=1}^{N} \log \left( \sum_{k=1}^K \pi_k \, \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right).
$$


## Ventajas y desventajas
- **Ventajas:** modela clusters elípticos, permite asignaciones suaves y captura varianzas diferentes.
- **Desventajas:** sensible a la inicialización, puede converger a óptimos locales y requiere elegir $K$.
- **Comparación con k-means:** k-means es más simple y rápido, pero GMM es más flexible en formas y covarianzas.


## Aplicaciones
- Segmentación de clientes o usuarios cuando las subpoblaciones se solapan.
- Modelado de densidad y detección de anomalías.
- Análisis de datos biológicos, financieros o de sensores con clusters no esféricos.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
# Crear datos sintéticos con tres clusters de distinta forma y dispersión
X, y_true = make_blobs(n_samples=500, centers=[[-3, -3], [0, 3], [3, 0]], cluster_std=[0.8, 0.6, 1.2], random_state=RANDOM_SEED)
df = pd.DataFrame(X, columns=['x1', 'x2'])
df['cluster_true'] = y_true
df.head()


In [ ]:
# Ajustar el modelo de mezcla gaussiana
gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=RANDOM_SEED, n_init=5)
gmm.fit(df[['x1', 'x2']])
labels = gmm.predict(df[['x1', 'x2']])
df['cluster_gmm'] = labels
print('Medias de componentes:')
print(gmm.means_)
print('\nPesos de mezcla:')
print(gmm.weights_)


In [ ]:
# Evaluación con el coeficiente de silhouette
score = silhouette_score(df[['x1', 'x2']], df['cluster_gmm'])
print(f'Silhouette score: {score:.3f}')

print(df['cluster_gmm'].value_counts().sort_index())


In [ ]:
def dibujar_elipse(ax, mean, cov, color):
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    width, height = 2 * np.sqrt(eigvals)
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle, edgecolor=color, facecolor='none', linewidth=2)
    ax.add_patch(ellipse)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(df['x1'], df['x2'], c=df['cluster_gmm'], cmap='viridis', s=35, alpha=0.7)
for k, (mean, cov) in enumerate(zip(gmm.means_, gmm.covariances_)):
    dibujar_elipse(ax, mean, cov, color=f'C{k}')
ax.set_title('Clustering con Gaussian Mixture Model')
ax.set_xlabel('x1')
ax.set_ylabel('x2')
legend1 = ax.legend(*scatter.legend_elements(), title='Cluster')
ax.add_artist(legend1)
ax.grid(True)
plt.show()


## Conclusiones
- Los Gaussian Mixture Models permiten modelar clusters con diferentes formas y covarianzas.
- El algoritmo EM alterna entre asignar pertenencias suaves ($r_{ik}$) y actualizar parámetros ($\mu_k$, $\Sigma_k$, $\pi_k$).
- En este ejemplo, la implementación de `scikit-learn` facilita el ajuste y la visualización de las elipses de covarianza.
- Es importante validar la cantidad de componentes $K$ con criterios como BIC, AIC o métricas de silhouette.


## Reproducibilidad
Instalar dependencias mínimas para ejecutar este notebook:
```bash
pip install numpy pandas scikit-learn matplotlib jupyter
```

Fijar `RANDOM_SEED = 42` asegura resultados reproducibles en el ejemplo sintético.
